<div align="center">

# Day 01: Foundations of Reinforcement Learning

**Reinforcement Learning and Language Model Alignment (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-01) · [Lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html) · [Interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lab.html) · [PDF notes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/Day01_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this notebook is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from itertools import product
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.4), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('environment ready')

## Python step 1: Values, variables, lists and decisions

A notebook is a sequence of cells. Text cells explain, and code cells hold Python instructions that run when the cell is executed with Shift+Enter, with the output shown below the cell &#91;[7](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-7)&#93;. This step plays a three-armed bandit with epsilon-greedy and scores it by its regret &#91;[5](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-5)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "check" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### Values, lists and variables

In [ ]:
win_rates = [0.30, 0.45, 0.40]   # true chance of a reward per arm, unknown to the agent
pulls = [0, 0, 0]                 # how often each arm was pulled
wins = [0, 0, 0]                  # how many rewards each arm gave
estimates = [0.0, 0.0, 0.0]       # the agent's estimate of each win rate
print(len(win_rates), "arms; best true win rate:", max(win_rates))

A list holds values in order between square brackets, and positions start at 0, so `win_rates[1]` is 0.45. `len` returns the length and `max` the largest element. The agent never reads `win_rates`; it only sees rewards.

### A decision: explore or exploit

In [ ]:
import random
rng = random.Random(0)          # a reproducible random number generator
eps = 0.1
if rng.random() < eps:
    arm, reason = rng.randrange(3), "explore"
else:
    arm, reason = estimates.index(max(estimates)), "exploit"
print(f"chose arm {arm} to {reason}")

`rng.random()` returns a number between 0 and 1, so the condition is true with probability `eps`. `rng.randrange(3)` returns 0, 1 or 2. `estimates.index(max(estimates))` finds the position of the largest estimate, the arm that looks best. Two values can be assigned at once, separated by a comma.

### A loop of 300 pulls

In [ ]:
for t in range(300):
    if rng.random() < eps:
        arm = rng.randrange(3)
    else:
        arm = estimates.index(max(estimates))
    reward = 1 if rng.random() < win_rates[arm] else 0
    pulls[arm] += 1
    wins[arm] += reward
    estimates[arm] = wins[arm] / pulls[arm]
print("pulls:", pulls)
print("estimates:", [round(e, 3) for e in estimates])

The loop runs its block 300 times. `1 if rng.random() < win_rates[arm] else 0` is a conditional expression: It gives the reward 1 with the win rate of the arm as probability, and 0 otherwise. `pulls[arm] += 1` adds one to an element of the list. The estimate of an arm is its share of wins so far, updated after every pull. The square brackets in the last line build a new list of rounded values, a list comprehension.

### A function for the regret

In [ ]:
def regret(pulls, win_rates):
    """Expected reward lost against always pulling the best arm."""
    best = max(win_rates)
    return sum(n * (best - p) for n, p in zip(pulls, win_rates))

print("regret after 300 pulls:", round(regret(pulls, win_rates), 2))

`def` defines a function with parameters and a `return` value, and the text in triple quotation marks documents it. `zip` walks two lists in step. Each pull of an arm that is worse than the best one adds the difference of their win rates to the regret. The function therefore computes the regret of the lecture, grouped by arm: $R_T = \sum_a N_a (\mu^* - \mu_a)$, with `pulls` for the numbers of pulls $N_a$, `win_rates` for the probabilities $\mu_a$ and `best` for $\mu^*$. In this run most pulls went to the arm with a win rate of 0.40 rather than the best arm with 0.45: The noisy early estimates favoured it, and ten percent of exploration did not correct them within 300 pulls.

### A dictionary of results

In [ ]:
summary = {"arms": 3, "epsilon": eps, "total reward": sum(wins),
           "regret": round(regret(pulls, win_rates), 2)}
for key, value in summary.items():
    print(f"{key:13s} {value}")

A dictionary maps keys to values between curly braces, and `items` returns key and value pairs for a loop. In an f-string, `{key:13s}` pads the text to thirteen characters so that the values line up.

**Quick check.** Why does an epsilon-greedy agent with `eps = 0` risk a large regret?

<details><summary>Answer</summary>

It never explores: If the first pulls make a poor arm look best, it keeps pulling that arm and never corrects the estimate of the others.

</details>

### Exercises for Python step 1

**Exercise 1.1.** How many pulls did the loop make in total? Store `sum(pulls)` in `n_pulls`.

In [ ]:
n_pulls = None   # your answer

check("Exercise 1.1", n_pulls, 300)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_pulls = sum(pulls)

check("Exercise 1.1", n_pulls, 300)

**Exercise 1.2.** Write a function `best_arm(estimates)` that returns the position of the largest estimate, and store `best_arm([0.2, 0.7, 0.5])` in `answer_12`.

In [ ]:
def best_arm(estimates):
    return None   # your code

answer_12 = best_arm([0.2, 0.7, 0.5])

check("Exercise 1.2", answer_12, 1)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def best_arm(estimates):
    return estimates.index(max(estimates))

answer_12 = best_arm([0.2, 0.7, 0.5])

check("Exercise 1.2", answer_12, 1)

**Exercise 1.3.** Use `regret` to store the regret of pulling each of the three arms ten times, `[10, 10, 10]`, in `answer_13`.

In [ ]:
answer_13 = None   # your answer

check("Exercise 1.3", answer_13, 2.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer_13 = regret([10, 10, 10], win_rates)

check("Exercise 1.3", answer_13, 2.0)

## 1. Bandits over thirty seeds &#91;[2](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-2), [3](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-3)&#93;

*Lecture: [Bandits: exploration against exploitation](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#bandits-exploration-against-exploitation)*

**Python move: NumPy arrays of runs.** Each strategy is run with thirty seeds, and the runs are stacked into an array with one row per seed, so that means and percentiles over seeds are single calls.

**From the formulas to the code.** In `run_bandit`, the arrays `N` and `S` hold the number of pulls $N_a$ and the number of paid pulls $S_a$ of every arm, so `S / N` is the estimate $\hat{\mu}_a = S_a / N_a$. Greedy and epsilon-greedy take `np.argmax` of it. UCB, the upper confidence bound, adds the bonus $c \sqrt{\ln t / N_a}$ in `c * np.sqrt(np.log(t + 1) / N)`. The code writes `t + 1` because Python counts the pulls from 0. Thompson sampling draws from $\mathrm{Beta}(1 + S_a, 1 + N_a - S_a)$ with `rng.beta(alpha, beta)`. The line `regret[t] = best - mus[a]` is one term $\mu^* - \mu_{a_t}$ of the regret, and `np.cumsum` adds the terms up to $R_T$. Until every arm has been pulled at least once, greedy, epsilon-greedy and UCB do not use their formulas, which divide by $N_a$: UCB pulls an arm that it has not tried yet, and the other two pull at random.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "answer_13" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def run_bandit(strategy, mus, T=1500, seed=0, eps=0.1, c=2.0):
    rng = np.random.default_rng(seed)
    K = len(mus)
    N = np.zeros(K); S = np.zeros(K)            # pulls and successes
    alpha = np.ones(K); beta = np.ones(K)       # Thompson posterior
    regret = np.zeros(T); best = mus.max()
    for t in range(T):
        if strategy == "epsilon-greedy":
            if rng.random() < eps or N.min() == 0:
                a = int(rng.integers(K))
            else:
                a = int(np.argmax(S / np.maximum(N, 1)))
        elif strategy == "UCB":
            if N.min() == 0:
                a = int(np.argmin(N))
            else:
                a = int(np.argmax(S / N + c * np.sqrt(np.log(t + 1) / N)))
        elif strategy == "Thompson":
            a = int(np.argmax(rng.beta(alpha, beta)))
        else:                                    # pure greedy, the control condition
            a = int(np.argmax(S / np.maximum(N, 1))) if N.min() > 0 else int(rng.integers(K))
        r = float(rng.random() < mus[a])
        N[a] += 1; S[a] += r
        alpha[a] += r; beta[a] += 1 - r
        regret[t] = best - mus[a]
    return np.cumsum(regret)

MUS = np.array([0.30, 0.35, 0.38, 0.40, 0.42, 0.45])     # deliberately close together
N_SEEDS = 30
curves = {}
for strat in ("greedy", "epsilon-greedy", "UCB", "Thompson"):
    runs = np.array([run_bandit(strat, MUS, seed=s) for s in range(N_SEEDS)])
    curves[strat] = runs

fig, ax = plt.subplots(figsize=(7, 3.8))
for strat, col in (("greedy", "#7f8c8d"), ("epsilon-greedy", "#c0392b"),
                   ("UCB", "#2980b9"), ("Thompson", "#27ae60")):
    R = curves[strat]
    m, lo, hi = R.mean(0), np.percentile(R, 10, 0), np.percentile(R, 90, 0)
    ax.plot(m, color=col, label=f"{strat}  (final {m[-1]:.0f})")
    ax.fill_between(np.arange(len(m)), lo, hi, color=col, alpha=.15)
ax.set_xlabel("pulls"); ax.set_ylabel("cumulative regret")
ax.set_title(f"Six arms, best = {MUS.max():.2f}, {N_SEEDS} seeds (band = 10th-90th percentile)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

print(pd.DataFrame({s: {"mean final regret": c[:, -1].mean(),
                        "sd across seeds": c[:, -1].std(),
                        "worst seed": c[:, -1].max()} for s, c in curves.items()}).T.round(1).to_string())

print("\nUCB with different values of c:")
for cc in (0.25, 0.5, 1.0, 2.0, 4.0):
    R = np.array([run_bandit("UCB", MUS, seed=s, c=cc) for s in range(N_SEEDS)])
    print(f"   c = {cc:4.2f}   mean final regret {R[:, -1].mean():6.1f}   "
          f"worst seed {R[:, -1].max():6.1f}")

**Python note.** `np.random.default_rng(seed)` creates a random number generator of its own. The same seed always gives the same sequence of random numbers, so every run can be repeated exactly. `np.argmax(x)` returns the position of the largest element of an array, and `np.maximum(N, 1)` replaces every 0 by 1, which avoids a division by zero for an arm that has not been pulled yet. Each array of `curves` holds one row per seed and one column per pull. `R.mean(0)` and `np.percentile(R, 10, 0)` work along axis 0, over the seeds, so they return one value per pull, and `c[:, -1]` takes the last column, the final regret of every seed. `plt.subplots` creates a figure with a set of axes, `ax`, on which the curves are drawn, and `figsize` gives its width and height in inches. The table comes from a dictionary comprehension, `{s: {...} for s, c in curves.items()}`, with one entry per strategy, and `.T` swaps its rows and columns.

**Observe and reflect.** Look at the bands of the figure. Greedy has the widest one: In some seeds it finds the best arm at once, and in others it stays with a poor arm for the whole run. What does such a band mean for someone who can afford only one run? Which column of the printed table would you read first if one bad run were costly?

The next cell finds the strategy with the lowest mean final regret. `min(curves, key=...)` walks over the keys of the dictionary and returns the key for which the function given as `key` is smallest. A `lambda` is a function written in one line: `lambda s: curves[s][:, -1].mean()` takes the name of a strategy and returns its mean final regret, the mean of the last column over the thirty seeds.

In [ ]:
best_strategy = min(curves, key=lambda s: curves[s][:, -1].mean())
print("lowest mean final regret:", best_strategy)

**Observe and reflect.** Is the strategy with the lowest mean final regret also the one with the lowest worst seed? Use the two tables of the section to explain how a method can win on the mean and lose on the worst seed.

## 2. TokenWorld as a decision process

*Lecture: [Markov decision processes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#markov-decision-processes)*

The environment of the week, with its reward written down.

**From the formulas to the code.** A state is a tuple of token numbers, the sentence so far, and an action is the number of the next token. `step(s, a)` is the transition: It appends the token, and its outcome is certain. `score_sequence` is the reward table of the lecture, and `is_terminal` decides whether an episode has ended.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "best_strategy" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# TokenWorld: sequence generation as a Markov decision process.
#
#   state     s_t = the tokens generated so far, the prefix
#   action    a_t = the next token, from the vocabulary
#   transition      s_{t+1} = s_t followed by a_t, deterministic
#   reward    r_t = 0 until the sequence ends, then a programmatic score
#   done      when <eos> is emitted or the length cap is reached
#
# The tiny version below has four tokens and four positions, so the whole MDP can be
# enumerated and solved exactly.

TINY_VOCAB = ["the", "cat", "sat", "<eos>"]     # 4 tokens
EOS = 3
MAXLEN = 4

def score_sequence(seq, vocab=TINY_VOCAB):
    """The true reward: hand-written, deterministic, known to the designer and hidden from the agent.

    A well-formed utterance in this toy language: starts with 'the', contains 'cat' exactly
    once, ends with '<eos>', and is short. Everything else is penalised.
    """
    toks = [vocab[i] for i in seq if vocab[i] != "<eos>"]
    r = 0.0
    if not toks:
        return -1.0                                    # empty utterance
    if toks[0] == "the":     r += 1.0
    else:                    r -= 0.5
    n_cat = toks.count("cat")
    if n_cat == 1:           r += 1.5
    elif n_cat == 0:         r -= 0.5
    else:                    r -= 0.8 * (n_cat - 1)     # repetition is penalised
    if "sat" in toks and "cat" in toks and toks.index("cat") < toks.index("sat"):
        r += 1.0                                       # word order matters
    r -= 0.25 * len(toks)                              # brevity bonus
    return float(r)


def enumerate_states(vocab_n=4, maxlen=MAXLEN):
    """Every reachable prefix. A prefix ending in EOS is terminal."""
    states = [()]
    frontier = [()]
    for _ in range(maxlen):
        nxt = []
        for s in frontier:
            if s and s[-1] == EOS:
                continue
            for a in range(vocab_n):
                nxt.append(s + (a,))
        states += nxt
        frontier = nxt
    return states


def is_terminal(s, maxlen=MAXLEN):
    return len(s) >= maxlen or (len(s) > 0 and s[-1] == EOS)


def step(s, a, maxlen=MAXLEN, vocab=TINY_VOCAB):
    """One transition. The reward is zero until the episode ends."""
    s2 = s + (a,)
    done = is_terminal(s2, maxlen)
    r = score_sequence(s2, vocab) if done else 0.0
    return s2, r, done


def rollout(policy_fn, rng, maxlen=MAXLEN, vocab_n=4):
    """Run one episode. `policy_fn(state) -> action probabilities`."""
    s, traj = (), []
    while not is_terminal(s, maxlen):
        p = policy_fn(s)
        a = int(rng.choice(len(p), p=p))
        s2, r, done = step(s, a, maxlen)
        traj.append((s, a, r))
        s = s2
    return traj, s

**Python note.** A state is a tuple of token numbers, such as `(0, 1)` for `the cat`. A tuple is written with round brackets and cannot be changed once it is made, so it can serve as a key of a dictionary, which a list cannot. `s + (a,)` builds a new, longer tuple, and the comma makes `(a,)` a tuple of one element. In `enumerate_states`, `_` is the usual name for a loop variable that is not used, and `continue` skips a sentence that has already ended, so that no token follows `<eos>`. The `while` loop of `rollout` repeats its block as long as its condition is true, here until the sentence is complete.

In [ ]:
S = enumerate_states()
terminal = [s for s in S if is_terminal(s)]
print(f"vocabulary : {TINY_VOCAB}")
print(f"max length : {MAXLEN}")
print(f"states     : {len(S)}  ({len(terminal)} terminal)")
print(f"policies   : {4 ** len([s for s in S if not is_terminal(s)]):.3g} deterministic ones\n")

sc = np.array([score_sequence(s) for s in terminal])
print(f"reward over all {len(terminal)} complete sequences:"
      f"  min {sc.min():.2f}   mean {sc.mean():.2f}   max {sc.max():.2f}")

show = sorted(terminal, key=score_sequence, reverse=True)
print("\nbest five sequences:")
for s in show[:5]:
    print(f"   {score_sequence(s):+.2f}   {' '.join(TINY_VOCAB[i] for i in s)}")
print("worst three:")
for s in show[-3:]:
    print(f"   {score_sequence(s):+.2f}   {' '.join(TINY_VOCAB[i] for i in s)}")

**Python note.** `sorted(terminal, key=score_sequence, reverse=True)` orders the sequences by their reward, from the largest down. `show[:5]` takes the first five elements and `show[-3:]` the last three. `' '.join(...)` joins the words of a sequence into one text with a space between them, and `:+.2f` prints a number with its sign and two decimals.

**Observe and reflect.** The five best sentences differ in one or two tokens. Write the reward of `the cat sat <eos>` and of `the cat sat the` as sums of the rules of the reward table. Which rule makes the difference of 0.25?

## 3. The Windy Cliff and dynamic programming &#91;[4](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-4)&#93;

*Lecture: [Values and the Bellman equation](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#values-and-the-bellman-equation)*

**From the formulas to the code.** `P[(s, a)]` lists every outcome of action `a` in state `s` as a triple of probability, next state and reward: the terms $P(s' \mid s, a)$ and $r(s, a, s')$ of the Bellman equation. In the value iteration loop, the inner `sum(p * (r + GAMMA * Vv[s2]) ...)` is the action value $Q(s, a)$ and the `max` over the four actions is the backup $V(s) = \max_a Q(s, a)$. The variable `d` holds the largest change of a value in one sweep, and the loop stops when it falls below $10^{-9}$. The loop stores each new value at once, so later states of the same sweep already use it. `policy_evaluation` repeats the Bellman expectation equation for the current policy `pi`, and `greedy_policy` switches every state to the action with the highest action value: the two steps of policy iteration.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "show" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# Windy Cliff: a 4 x 8 gridworld with stochastic transitions.
#
# With probability `slip` the wind pushes the agent sideways. Stepping into the cliff
# costs 100 and returns the agent to the start; every other step costs 1.

ROWS, COLS = 4, 8
START, GOAL = (3, 0), (3, 7)
CLIFF = [(3, c) for c in range(1, 7)]
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]     # up, down, left, right
ANAMES = ["up", "down", "left", "right"]

def grid_step(s, a, slip=0.15, rng=None):
    """With probability `slip` the wind pushes the agent sideways instead."""
    if rng is not None and rng.random() < slip:
        a = int(rng.choice([2, 3]))
    dr, dc = ACTIONS[a]
    r, c = s[0] + dr, s[1] + dc
    r = min(max(r, 0), ROWS - 1); c = min(max(c, 0), COLS - 1)
    s2 = (r, c)
    if s2 in CLIFF:
        return START, -100.0, False       # fall off; back to the start, episode continues
    if s2 == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def grid_transitions(slip=0.15):
    """Exact P(s'|s,a) and R(s,a,s') for dynamic programming."""
    P = {}
    for r in range(ROWS):
        for c in range(COLS):
            s = (r, c)
            if s == GOAL:
                continue
            for a in range(4):
                outcomes = {}
                for real_a, prob in [(a, 1 - slip), (2, slip / 2), (3, slip / 2)]:
                    dr, dc = ACTIONS[real_a]
                    rr = min(max(r + dr, 0), ROWS - 1); cc = min(max(c + dc, 0), COLS - 1)
                    s2 = (rr, cc)
                    if s2 in CLIFF:
                        s2, rew = START, -100.0
                    elif s2 == GOAL:
                        rew = 0.0
                    else:
                        rew = -1.0
                    key = (s2, rew)
                    outcomes[key] = outcomes.get(key, 0.0) + prob
                P[(s, a)] = [(p, s2, rew) for (s2, rew), p in outcomes.items()]
    return P

**Python note.** `P` is a dictionary whose keys are pairs `(s, a)` and whose values are lists of triples `(p, s2, rew)`: the probability, the next state and the reward of each outcome. `outcomes.get(key, 0.0)` returns the stored value, or 0.0 for a key that is not there yet, so outcomes that lead to the same cell add up their probabilities.

In [ ]:
P = grid_transitions(slip=0.15)
STATES = [(r, c) for r in range(ROWS) for c in range(COLS)]
GAMMA = 0.99

def policy_evaluation(pi, P, theta=1e-9):
    V = {s: 0.0 for s in STATES}
    for _ in range(10000):
        d = 0.0
        for s in STATES:
            if s == GOAL:
                continue
            v = sum(pi[s][a] * sum(p * (r + GAMMA * V[s2]) for p, s2, r in P[(s, a)])
                    for a in range(4))
            d = max(d, abs(v - V[s])); V[s] = v
        if d < theta:
            break
    return V

def greedy_policy(V, P):
    pi = {}
    for s in STATES:
        if s == GOAL:
            pi[s] = np.ones(4) / 4; continue
        q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in P[(s, a)]) for a in range(4)]
        pi[s] = np.eye(4)[int(np.argmax(q))]
    return pi

# --- policy iteration
pi = {s: np.ones(4) / 4 for s in STATES}
for k in range(50):
    V = policy_evaluation(pi, P)
    pi_new = greedy_policy(V, P)
    if all(np.array_equal(pi[s], pi_new[s]) for s in STATES):
        break
    pi = pi_new
print(f"policy iteration converged after {k} improvement steps")
print(f"V(start) = {V[START]:.2f}\n")

# --- value iteration
Vv = {s: 0.0 for s in STATES}
for it in range(10000):
    d = 0.0
    for s in STATES:
        if s == GOAL:
            continue
        best = max(sum(p * (r + GAMMA * Vv[s2]) for p, s2, r in P[(s, a)]) for a in range(4))
        d = max(d, abs(best - Vv[s])); Vv[s] = best
    if d < 1e-9:
        break
print(f"value iteration converged in {it} sweeps,  V(start) = {Vv[START]:.2f}")
print(f"the two agree to {max(abs(V[s] - Vv[s]) for s in STATES):.2e}")

**Python note.** `sum(p * (r + GAMMA * V[s2]) for p, s2, r in P[(s, a)])` is the inner sum of the Bellman equation in one line: The expression before `for` is computed for every outcome of the action, and `sum` adds the results. Such an expression in round brackets is called a generator expression. `{s: 0.0 for s in STATES}` is a dictionary comprehension: It builds a dictionary with the value 0.0 for every state. `break` leaves the loop over the sweeps as soon as no value has changed by more than `theta`.

**Exercise A.** Store the value of the start state found by value iteration, rounded to two decimals, in `v_start`.

In [ ]:
v_start = None   # your answer

check("Exercise A", v_start, round(Vv[START], 2))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
v_start = round(Vv[START], 2)

check("Exercise A", v_start, round(Vv[START], 2))

In [ ]:
arrows = {0: "^", 1: "v", 2: "<", 3: ">"}
grid = np.full((ROWS, COLS), " ", dtype=object)
val = np.zeros((ROWS, COLS))
for s in STATES:
    val[s] = Vv[s]
    grid[s] = "G" if s == GOAL else ("X" if s in CLIFF else arrows[int(np.argmax(pi[s]))])
grid[START] = "S"

fig, ax = plt.subplots(figsize=(7.5, 3.0))
im = ax.imshow(val, cmap="RdYlGn")
for r in range(ROWS):
    for c in range(COLS):
        ax.text(c, r, f"{grid[r, c]}\n{val[r, c]:.0f}", ha="center", va="center", fontsize=8)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Windy Cliff: optimal policy and state values  (S start, G goal, X cliff)")
plt.colorbar(im, ax=ax, shrink=.8); plt.tight_layout(); plt.show()

Change `slip` to 0.0 in `grid_transitions` above and run the two cells again: The route stays the same, along the row next to the cliff, because the wind only pushes to the left or to the right. What changes is the value of the start, from -17.89 to -7.73: The difference is the price of the wind, most of it paid at the start, where a push to the right ends in the cliff.

**Observe and reflect.** Set `slip=0.3` and run the two cells again. Does the route of the arrows change? In which cells does the value fall most, and what do these cells have in common?

## 4. The exact optimum of TokenWorld

*Lecture: [TokenWorld, stage 1: an exact optimum as a yardstick](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#tokenworld-stage-1-an-exact-optimum-as-a-yardstick)*

**From the formulas to the code.** `value_iteration_tokenworld` applies the same backup with $\gamma = 1$. A transition of TokenWorld is certain, so the sum over next states has a single term, `r + gamma * V[s2]`. `greedy_action` returns the four action values $Q^*(s, a)$ of a state and the token with the largest one, $\pi^*(s) = \arg\max_a\, Q^*(s, a)$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "grid" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def value_iteration_tokenworld(gamma=1.0):
    V = {s: 0.0 for s in S}
    sweeps = 0
    for _ in range(MAXLEN + 2):
        delta = 0.0
        for s in S:
            if is_terminal(s):
                continue
            best = -1e9
            for a in range(len(TINY_VOCAB)):
                s2, r, done = step(s, a)
                best = max(best, r + (0.0 if done else gamma * V[s2]))
            delta = max(delta, abs(best - V[s])); V[s] = best
        sweeps += 1
        if delta < 1e-12:
            break
    return V, sweeps

V_star, sweeps = value_iteration_tokenworld()
print(f"converged in {sweeps} sweeps (the horizon is {MAXLEN}, as the theory predicts)")
print(f"V*(empty prefix) = {V_star[()]:.4f}   <- the best achievable expected return\n")

def greedy_action(s, V):
    q = []
    for a in range(len(TINY_VOCAB)):
        s2, r, done = step(s, a)
        q.append(r + (0.0 if done else V[s2]))
    return int(np.argmax(q)), np.array(q)

s, out = (), []
print("the optimal policy, unrolled:")
while not is_terminal(s):
    a, q = greedy_action(s, V_star)
    prefix = " ".join(TINY_VOCAB[i] for i in s) or "(empty)"
    print(f"   after {prefix:22s} Q = " +
          "  ".join(f"{TINY_VOCAB[i]}:{q[i]:+.2f}" for i in range(4)) +
          f"   -> {TINY_VOCAB[a]}")
    out.append(a); s = s + (a,)
print(f"\nresult: '{' '.join(TINY_VOCAB[i] for i in out)}'   reward {score_sequence(tuple(out)):.2f}")

**Observe and reflect.** Value iteration needed 26 sweeps on the Windy Cliff and only a handful on TokenWorld. TokenWorld has no loops: Every action makes the sentence longer, and no sentence has more than four tokens. How does that limit the number of sweeps after which no value can change any more?

In [ ]:
# The return of a policy that does not plan: uniformly random actions.
rng = np.random.default_rng(RANDOM_STATE)
uniform = lambda s: np.ones(4) / 4
rand_returns = np.array([rollout(uniform, rng)[0][-1][2] for _ in range(4000)])

print(f"random policy      mean return {rand_returns.mean():+.3f}   "
      f"best-of-4000 {rand_returns.max():+.2f}")
print(f"optimal policy     mean return {V_star[()]:+.3f}")
print(f"fraction of random rollouts that are optimal: "
      f"{(rand_returns >= V_star[()] - 1e-9).mean():.3%}")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(rand_returns, bins=40, color="#7f8c8d", alpha=.85, label="random policy")
ax.axvline(V_star[()], color="#c0392b", lw=2, label=f"optimal = {V_star[()]:.2f}")
ax.axvline(rand_returns.mean(), color="#2980b9", lw=2, ls="--",
           label=f"random mean = {rand_returns.mean():.2f}")
ax.set_xlabel("episode return"); ax.set_ylabel("count"); ax.legend(fontsize=8)
ax.set_title("The gap that learning has to close")
plt.tight_layout(); plt.show()

**Exercise B.** Compute the probability that a uniformly random policy produces the optimal sequence, four specific tokens out of four each, and store it in `p_opt`. Compare it with the fraction measured above.

In [ ]:
p_opt = None   # your answer

check("Exercise B", p_opt, 1 / 256)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_opt = (1 / 4) ** 4

check("Exercise B", p_opt, 1 / 256)

## 5. Your field: a bandit problem

*Lecture: [Your field](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#your-field)*

Change `FIELD` to `marketing`, `energy`, `agriculture` or `education` and run the cell again.

**From the formulas to the code.** `bandit_study` calls `run_bandit` of section 1 on the four options of the chosen field, so the estimates, the UCB bonus, the Beta beliefs and the regret are computed by the same lines.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "p_opt" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
FIELD = "medicine"     # choose: medicine, marketing, energy, agriculture or education

BANDIT_FIELDS = {
    "medicine":    (["dose 1", "dose 2", "dose 3", "dose 4"], [0.20, 0.35, 0.50, 0.45], "patient responds"),
    "marketing":   (["subject A", "subject B", "subject C", "subject D"], [0.04, 0.05, 0.07, 0.06], "email opened"),
    "energy":      (["tariff A", "tariff B", "tariff C", "tariff D"], [0.30, 0.25, 0.40, 0.35], "driver accepts"),
    "agriculture": (["schedule A", "schedule B", "schedule C", "schedule D"], [0.55, 0.60, 0.50, 0.65], "yield above target"),
    "education":   (["hint A", "hint B", "hint C", "hint D"], [0.40, 0.55, 0.50, 0.45], "problem solved"),
}

def bandit_study(field, T=1000, seeds=20):
    """Cumulative regret of four strategies on the arms of one field, over several seeds."""
    names, mus, success = BANDIT_FIELDS[field]; mus = np.array(mus)
    out = {s: np.array([run_bandit(s, mus, T=T, seed=k) for k in range(seeds)]) for s in ("greedy", "epsilon-greedy", "UCB", "Thompson")}
    return names, mus, success, out

f_names, f_mus, f_success, f_curves = bandit_study(FIELD)
field_regret = pd.DataFrame({s: [c[:, -1].mean(), c[:, -1].std(), c[:, -1].max()] for s, c in f_curves.items()},
                            index=["mean final regret", "spread (SD)", "worst seed"]).T
print(f"{FIELD}: best arm {f_names[int(np.argmax(f_mus))]} ({f_success} with probability {f_mus.max():.2f})")
print(field_regret.round(1).to_string())
plt.figure(figsize=(7, 3.4))
for s, c in f_curves.items():
    plt.plot(c.mean(0), label=s); plt.fill_between(np.arange(c.shape[1]), np.percentile(c, 10, 0), np.percentile(c, 90, 0), alpha=.12)
plt.xlabel("round"); plt.ylabel("cumulative regret"); plt.title(f"{FIELD}: mean and 10 to 90 percent band over 20 seeds", fontsize=9)
plt.legend(fontsize=8); plt.show()

**Python note.** `c.shape` is the pair of the numbers of rows and columns of the array `c`, so `np.arange(c.shape[1])` numbers its columns, the rounds.

`idxmin()` returns the label of the row in which a column of a pandas table is smallest, here the name of the strategy with the lowest mean final regret in your field.

In [ ]:
best_strategy = field_regret["mean final regret"].idxmin()
print(FIELD, "->", best_strategy)

**Observe and reflect.** Compare the ranking of your field with the ranking on the six close arms of section 1. Are the arms of your field further apart or closer together, and how does that change what exploration is worth?

## 6. Going further (optional): seeds, spread and a cheap correctness check &#91;[6](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#ref-6)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lecture.html#going-further-optional)*

The last row of the table, with epsilon equal to 1, is the uniform random policy: Its mean return must match the value measured in section 4 by a different route.

**From the formulas to the code.** `study` returns the mean $\bar{x}$ with `finals.mean()`, the standard deviation with `finals.std()` and the 10th and 90th percentiles with `np.percentile`. `exact_random` is the expected return of the random policy from the lecture: the reward of every finished sentence times its probability $(1/4)^{|x|}$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "field_regret" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/rl-alignment-veltech-NB-lecture/main/days/day-01/NB01_foundations_of_rl.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def evaluate_policy(policy_fn, n=400, seed=0):
    rng = np.random.default_rng(seed)
    return np.array([rollout(policy_fn, rng)[0][-1][2] for _ in range(n)])

def study(make_policy, n_seeds=15, n_eval=400, label=""):
    """Rule 1: many seeds.  Rule 2: report the spread together with the mean.
       Rule 3: fix the evaluation protocol.  Rule 4: state the sample size."""
    finals = []
    for s in range(n_seeds):
        pol = make_policy(s)
        finals.append(evaluate_policy(pol, n=n_eval, seed=1000 + s).mean())
    finals = np.array(finals)
    lo, hi = np.percentile(finals, [10, 90])
    return dict(policy=label, seeds=n_seeds, mean=finals.mean(), sd=finals.std(),
                p10=lo, p90=hi, best_seed=finals.max(), worst_seed=finals.min())

def eps_greedy_of_optimal(eps):
    def make(seed):
        rng_local = np.random.default_rng(seed)
        def pol(s):
            a, _ = greedy_action(s, V_star)
            p = np.full(4, eps / 4); p[a] += 1 - eps
            return p
        return pol
    return make

rows = [study(eps_greedy_of_optimal(e), label=f"eps-greedy(optimal), eps={e}")
        for e in (0.0, 0.1, 0.3, 0.6, 1.0)]
res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False))

**Python note.** `eps_greedy_of_optimal(eps)` returns a function, `make`, which itself returns a function, `pol`. Each inner function remembers the variables of the function around it, here `eps`. Such a function is called a closure. It lets `study` build a new policy for every seed without global variables.

In [ ]:
exact_random = sum(0.25 ** len(s) * score_sequence(s) for s in S if is_terminal(s))
print(f"exact expected return of the uniform policy : {exact_random:.3f}")
print(f"estimate from 4000 rollouts (section 4)       : {rand_returns.mean():.3f}")
print(f"estimate from the table, eps = 1.0            : {res.iloc[-1]['mean']:.3f}")

**Python note.** `res.iloc[-1]` selects the last row of the table `res` by its position, and `['mean']` the entry of that row in the column `mean`. In the next cell, `W.interact` draws a slider for every argument of `explore` and runs the function again whenever a slider moves.

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def explore(eps=0.1, c=2.0, seeds=10):
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for strat, col in (("epsilon-greedy", "#c0392b"), ("UCB", "#2980b9"), ("Thompson", "#27ae60")):
        R = np.array([run_bandit(strat, MUS, seed=s, eps=eps, c=c) for s in range(seeds)])
        ax.plot(R.mean(0), color=col, label=f"{strat}: {R[:, -1].mean():.1f}")
    ax.set_xlabel("pulls"); ax.set_ylabel("mean cumulative regret"); ax.legend(fontsize=8); plt.show()
W.interact(explore, eps=W.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.01),
           c=W.FloatSlider(value=2.0, min=0.1, max=4.0, step=0.1), seeds=W.IntSlider(value=10, min=3, max=30));

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement the decaying schedule epsilon = 1 / sqrt(t) and compare its regret with a fixed epsilon over thirty seeds. |
| Electronics and Communication Engineering | Model channel selection in a wireless network as a bandit in which the success rates drift slowly, and test whether a sliding-window estimate helps. |
| Electrical and Electronics Engineering | Formulate the charging of a battery over a day as a decision process with a time-varying electricity price, and solve a small version by value iteration. |
| Mechanical Engineering | Add a wall to the Windy Cliff and show how the optimal policy and the value of the start state change. |
| Biomedical Engineering | Treat the choice between three treatment protocols in an adaptive trial as a bandit and discuss why the worst-seed column matters more than the mean there. |
| Civil Engineering | Formulate the inspection schedule of a bridge as a Markov decision process with three condition states and solve it by policy iteration. |
| Aeronautical Engineering | Let the wind push the agent down, towards the cliff, instead of sideways, and find the slip probability at which the optimal route moves one row away from the edge and the one at which it moves two rows away. |
| Biotechnology | Extend TokenWorld with a fifth token and recompute the exact optimum and the probability that a random policy finds it. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-01/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-01).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>